# Chronological operation, commitment, and offshore benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/equinor/neqsim/blob/master/examples/notebooks/energy_networks/04_time_series_commitment_offshore_benchmark.ipynb)

This notebook demonstrates chronological profiles, integrated energy/cost/emissions KPIs, generator commitment constraints, and the reproducible offshore wind–gas benchmark.

**Implementation dependencies:** PRs #2614, #2617, and #2618.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = [Path(env_root).resolve()] if env_root else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")

PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_classes(neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True))
JClass = ns.JClass
print("NeqSim workspace classes loaded")

In [ ]:
EnergyTimeSeriesProfile = JClass("neqsim.process.equipment.energy.EnergyTimeSeriesProfile")
EnergyTimeSeriesSimulator = JClass("neqsim.process.equipment.energy.EnergyTimeSeriesSimulator")
CommittedEnergyGenerator = JClass("neqsim.process.equipment.energy.CommittedEnergyGenerator")
OffshoreEnergyReferenceCase = JClass("neqsim.process.equipment.energy.OffshoreEnergyReferenceCase")
EnergyBus = JClass("neqsim.process.equipment.stream.EnergyBus")
EnergyPort = JClass("neqsim.process.equipment.stream.EnergyPort")
EnergyType = JClass("neqsim.process.equipment.stream.EnergyType")
EnergyPortDirection = JClass("neqsim.process.equipment.stream.EnergyPortDirection")
EnergyPortMode = JClass("neqsim.process.equipment.stream.EnergyPortMode")
ProcessSystem = JClass("neqsim.process.processmodel.ProcessSystem")
UUID = JClass("java.util.UUID")

## 2. Run the published offshore benchmark

In [ ]:
result = OffshoreEnergyReferenceCase.run24HourCase()
print(result.toJson())

In [ ]:
import pandas as pd
interval_rows = []
for interval in result.getIntervals():
    report = interval.getNetworkReports().get(0)
    allocations = {a.getParticipantName(): a.getAllocatedPower()/1e6 for a in report.getAllocations()}
    interval_rows.append({
        "start_h": interval.getStartTimeSeconds()/3600.0,
        "duration_h": interval.getDurationSeconds()/3600.0,
        "served_MW": report.getServedDemand()/1e6,
        "unmet_MW": report.getUnmetDemand()/1e6,
        "curtailed_MW": report.getCurtailedSupply()/1e6,
        "wind_MW": allocations.get("offshore wind:power", 0.0),
        "gas_MW": allocations.get("gas turbine generation:power", 0.0),
        "critical_MW": allocations.get("critical process load:power", 0.0),
        "flexible_MW": allocations.get("flexible process load:power", 0.0),
        "cost_per_h": report.getOperatingCostPerHour(),
        "co2_kg_per_h": report.getCo2EmissionRate(),
    })
interval_df = pd.DataFrame(interval_rows)
interval_df

In [ ]:
import matplotlib.pyplot as plt
ax = interval_df.plot(x="start_h", y=["served_MW", "unmet_MW", "curtailed_MW"],
                      drawstyle="steps-post", marker="o", figsize=(9,4))
ax.set_xlabel("Time (h)")
ax.set_ylabel("Power (MW)")
ax.set_title("Offshore benchmark energy balance")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** The reference case serves all demand while producing deliberate wind curtailment in one interval. The fixed result is intended for regression and audit.

In [ ]:
ax = interval_df.plot(x="start_h", y=["cost_per_h", "co2_kg_per_h"],
                      drawstyle="steps-post", marker="o", figsize=(9,4))
ax.set_xlabel("Time (h)")
ax.set_ylabel("Rate per hour")
ax.set_title("Chronological operating cost and CO₂")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** Cost and emissions follow accepted gas generation, not total offered generation. Renewable curtailment is reported separately.

## 3. Demonstrate commitment constraints

In [ ]:
generator = CommittedEnergyGenerator("backup gas turbine", EnergyType.ELECTRICAL)
generator.setPowerLimits(5.0e6, 15.0e6)
generator.setRampRates(2.0e6, 3.0e6)
generator.setMinimumUpDownTimes(2.0*3600.0, 1.0*3600.0)
generator.setStartupPenalty(5000.0, 1200.0)
generator.initializeCommitment(False, 2.0*3600.0, 0.0)

requests_MW = [0, 8, 14, 6, 0, 0]
commit_rows = []
for hour, request in enumerate(requests_MW):
    generator.setRequestedPower(request*1e6)
    generator.runTransient(3600.0, UUID.randomUUID())
    step = generator.getLastStepResult()
    commit_rows.append({
        "hour": hour,
        "request_MW": request,
        "actual_MW": step.getGeneratedPower()/1e6,
        "online": int(step.isCommitted()),
        "blocked_start": int(step.isStartBlocked()),
        "blocked_stop": int(step.isStopBlocked()),
        "startup_count": generator.getStartupCount(),
    })
commit_df = pd.DataFrame(commit_rows)
commit_df

In [ ]:
ax = commit_df.plot(x="hour", y=["request_MW", "actual_MW"], marker="o", figsize=(9,4))
ax.set_xlabel("Time step (h)")
ax.set_ylabel("Power (MW)")
ax.set_title("Committed generator request and realized output")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** Realized output respects commitment state, minimum stable generation, ramp limits, and minimum up/down times. Startup penalties can be added to chronological economics.

## 4. Published benchmark KPIs

In [ ]:
kpis = pd.Series({
    "served energy (MWh)": result.getServedEnergyMWh(),
    "unmet energy (MWh)": result.getUnmetEnergyMWh(),
    "curtailed energy (MWh)": result.getCurtailedEnergyMWh(),
    "operating cost": result.getOperatingCost(),
    "CO2-equivalent (kg)": result.getCo2EmissionsKg(),
})
kpis

## Summary

The notebook demonstrates how steady-state process/energy calculations become a chronological operating study. Professional extensions may add stochastic wind scenarios, maintenance availability, battery degradation, reserve criteria, and mixed-integer scheduling.